--Start Spark

In [2]:
import os

print(os.environ["HADOOP_HOME"])

C:\hadoop


In [3]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# Initialize a Spark Session
spark = SparkSession.builder \
    .appName("PySpark pipeline") \
    .getOrCreate()

# Verify the session is working
print("Spark Version:", spark.version)

Spark Version: 3.5.6


In [4]:
import os

print(os.environ.get("HADOOP_HOME"))

C:\hadoop


 --Read data from files (CSV, Parquet) with proper schema handling.

In [5]:
from pyspark.sql.types import *

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("category", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("region", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("rating", IntegerType(), True)
])

df = spark.read.csv(
    "dataset.csv",
    header=True,
    schema=schema
)

In [6]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- rating: integer (nullable = true)



In [7]:
df.show(5)

+--------+-----------+-------------+---+------+-----------+------------+--------+------+------+----------+--------------+------+
|order_id|customer_id|customer_name|age|gender|   category|product_name|quantity| price|region|order_date|payment_method|rating|
+--------+-----------+-------------+---+------+-----------+------------+--------+------+------+----------+--------------+------+
|    1001|       C001|        Alice| 25|Female|Electronics|  Headphones|       2|1500.0| North|03-01-2025|           UPI|     4|
|    1002|       C002|          Bob| 32|  Male|   Clothing|     T-Shirt|       3| 700.0| South|03-01-2025|          Card|     5|
|    1003|       C003|      Charlie| 28|  Male|    Grocery|    Rice Bag|       1|1200.0|  East|04-01-2025|           UPI|     4|
|    1004|       C004|        David| 45|  Male|Electronics|    Keyboard|       1|1800.0|  West|05-01-2025|          Card|     5|
|    1005|       C005|          Eva| 22|Female|     Beauty|   Face Wash|       2| 350.0| North|05

Perform filtering and selection of required columns.

In [8]:
df = df.select(
    "order_id",
    "customer_id",
    "category",
    "quantity",
    "price",
    "region",
    "order_date"
)

In [9]:
# Only products whose price is greater than 500
df = df.filter(df.price > 500)
df.show()

+--------+-----------+-----------+--------+-------+------+----------+
|order_id|customer_id|   category|quantity|  price|region|order_date|
+--------+-----------+-----------+--------+-------+------+----------+
|    1001|       C001|Electronics|       2| 1500.0| North|03-01-2025|
|    1002|       C002|   Clothing|       3|  700.0| South|03-01-2025|
|    1003|       C003|    Grocery|       1| 1200.0|  East|04-01-2025|
|    1004|       C004|Electronics|       1| 1800.0|  West|05-01-2025|
|    1006|       C006|   Clothing|       1| 1800.0| South|06-01-2025|
|    1008|       C008|Electronics|       2|  800.0|  West|07-01-2025|
|    1010|       C010|   Clothing|       1| 2500.0| South|08-01-2025|
|    1012|       C012|Electronics|       1| 3200.0|  West|09-01-2025|
|    1013|       C013|     Beauty|       1| 1500.0| North|09-01-2025|
|    1014|       C014|   Clothing|       1| 2200.0| South|10-01-2025|
|    1016|       C016|Electronics|       1| 9500.0|  West|11-01-2025|
|    1020|       C02

Modify DataFrames (rename columns, cast data types, add new columns).

In [10]:
df = df.withColumnRenamed(
    "price",
    "product_price"
)

In [11]:
# Cast Date
from pyspark.sql.functions import to_timestamp

df = df.withColumn(
    "order_date",
    to_timestamp("order_date","dd-MM-yyyy")
)

In [12]:
# Adding new column
from pyspark.sql.functions import col

df = df.withColumn(
    "total_amount",
    col("quantity")*col("product_price")
)

Apply transformations and actions appropriately. Understand wide  transformations and performance concepts (Shuffle, Predicate Pushdown). Work with different file formats (CSV vs Parquet) and their impact on performance. Handle null values and filter datasets efficiently. Build data pipelines (read → transform → filter → write). Save processed data into required formats (CSV/Parquet). Follow best practices for large datasets (avoid collect(), use show())

In [13]:
# Handling null prices--it will become 0
df = df.na.fill(
    0,
    subset=["product_price"]
)

In [14]:
# Remove duplicates 
df = df.dropDuplicates()

In [15]:
# Group By transformation
from pyspark.sql.functions import sum

result = df.groupBy(
    "category"
).agg(
    sum("total_amount").alias("total_revenue")
)

In [16]:
result.show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|    Grocery|       3600.0|
|Electronics|     148000.0|
|   Clothing|      14400.0|
|     Beauty|       2200.0|
+-----------+-------------+



In [17]:
print(type(result))

<class 'pyspark.sql.dataframe.DataFrame'>


In [18]:
# Save csv file 
result.write.option("header", True).mode("overwrite").csv("output_in_csv")

In [19]:
result.write.mode("overwrite").parquet(
    "output_parquet"
)

Key Insights:
    
Used explicit schema instead of inferSchema=True.
    
Used show() instead of collect().
    
groupBy() causes a shuffle (wide transformation).
    
Filtering early reduces data processed.
    
Parquet is faster than CSV because it is a columnar format and supports predicate pushdown.
    
DataFrames are immutable, so every transformation returns a new DataFrame.